# Text classification with TF-IDF and Naive Bayes

Turn text into numeric features and fit a Multinomial Naive Bayes baseline. This dataset consists of 32 handcrafted sentences about weather or sports. It is deliberately small and cannot estimate performance on real messages.

The vocabulary must be learned inside the training pipeline. No network access or GPU is required.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

weather = [
    'Heavy rain is forecast this evening', 'The weather forecast predicts strong wind',
    'Snow covered the roads overnight', 'A cold storm will bring heavy snow',
    'The temperature rises under sunny skies', 'Warm weather and sunshine are expected',
    'Thunder and rain interrupted the afternoon', 'Storm clouds gathered above the city',
    'Fog reduced visibility on the road', 'Morning fog followed a cool night',
    'Wind pushed the storm toward the coast', 'Rain and wind continued through the night',
    'The forecast shows a drop in temperature', 'A sunny afternoon replaced the cloudy morning',
    'The weather report warns of snow', 'Cloudy skies may produce light rain',
]
sports = [
    'The team won the football match', 'A player scored the winning goal',
    'The coach prepared the team for the match', 'Fans celebrated a football victory',
    'The tournament final attracted many fans', 'The tennis player won the final set',
    'A strong serve decided the tennis match', 'The coach praised the players after victory',
    'The basketball team defended its lead', 'The player scored points in the final minute',
    'The football goalkeeper saved the shot', 'A tournament match ended in a draw',
    'The team trained before the tournament', 'Fans watched the winning player celebrate',
    'The coach discussed the final score', 'The tennis tournament begins on Monday',
]
texts = np.array(weather + sports)
labels = np.array(['weather'] * len(weather) + ['sports'] * len(sports))
train_text, test_text, train_y, test_y = train_test_split(texts, labels,
    test_size=0.25, stratify=labels, random_state=42)


## Fit vectorization and classification together

TF-IDF downweights words that appear across many documents. Multinomial Naive Bayes accepts these nonnegative features. Its independence assumption is a modeling simplification, not a statement that words are independent in language.


In [ ]:
pipeline = Pipeline([('text', TfidfVectorizer()), ('model', MultinomialNB())])
search = GridSearchCV(pipeline, {'model__alpha': [0.1, 0.5, 1.0]},
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='accuracy', n_jobs=1)
search.fit(train_text, train_y)
print('Best training-fold setting:', search.best_params_)
print(classification_report(test_y, search.predict(test_text), zero_division=0))


In [ ]:
new_text = ['Rain and wind are expected tonight', 'The player scored a goal',
            'The coach waited under cloudy skies']
display(pd.DataFrame({'text': new_text, 'prediction': search.predict(new_text)}))
print('Class order:', search.best_estimator_.classes_)
print('Vocabulary size:', len(search.best_estimator_.named_steps['text'].vocabulary_))
assert np.allclose(search.predict_proba(new_text).sum(axis=1), 1.0)


## Practice

The mixed sentence shows why keyword examples are limited. Add more varied, independently written sentences and examine errors. For a real dataset, handle duplicate documents, domain shifts, labels, and an independent test split before reporting a benchmark.

**References:** [text feature extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html).
